# Homework #4: F1 Model Building and MLflow Tracking

This notebook builds a Formula 1 podium-finish classifier and tracks the model runs with MLflow. The target is whether a driver finishes in the top 3. The feature set uses starting grid position, race timing variables, and historical driver/constructor performance calculated only from prior races.

## 1. Load the F1 data

The notebook is written for Databricks. If your course volume is mounted in the standard location, the path below should work without changes.

In [0]:
import os
import tempfile

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from mlflow.models import infer_signature
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)

base_path = "/Volumes/gr5069/raw/f1_data/"

results = pd.read_csv(base_path + "results.csv", na_values=["\\N"])
races = pd.read_csv(base_path + "races.csv", na_values=["\\N"])
drivers = pd.read_csv(base_path + "drivers.csv", na_values=["\\N"])
constructors = pd.read_csv(base_path + "constructors.csv", na_values=["\\N"])

print(results.shape, races.shape, drivers.shape, constructors.shape)

## 2. Build the modeling table

The target is `podium`, equal to 1 when `positionOrder <= 3`. To keep the features realistic, the driver and constructor history variables are shifted so each row only sees information available before that race.

In [0]:
df = (
    results.merge(races[["raceId", "year", "round", "name", "date"]], on="raceId", how="left")
    .merge(drivers[["driverId", "driverRef"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "constructorRef"]], on="constructorId", how="left")
)

numeric_cols = ["grid", "positionOrder", "fastestLapSpeed", "rank"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["raceId", "driverId", "constructorId", "grid", "positionOrder", "year", "round"])
df = df.sort_values(["year", "round", "raceId", "positionOrder"]).reset_index(drop=True)

df["podium"] = (df["positionOrder"] <= 3).astype(int)
df["grid"] = df["grid"].clip(lower=0)
df["started_from_pit_or_back"] = (df["grid"] == 0).astype(int)

df["driver_starts_before"] = df.groupby("driverId").cumcount()
df["constructor_starts_before"] = df.groupby("constructorId").cumcount()

df["driver_prior_avg_finish"] = df.groupby("driverId")["positionOrder"].transform(
    lambda s: s.shift().expanding().mean()
)
df["driver_prior_podium_rate"] = df.groupby("driverId")["podium"].transform(
    lambda s: s.shift().expanding().mean()
)
df["constructor_prior_avg_finish"] = df.groupby("constructorId")["positionOrder"].transform(
    lambda s: s.shift().expanding().mean()
)
df["constructor_prior_podium_rate"] = df.groupby("constructorId")["podium"].transform(
    lambda s: s.shift().expanding().mean()
)

history_fill_values = {
    "driver_prior_avg_finish": df["positionOrder"].mean(),
    "driver_prior_podium_rate": df["podium"].mean(),
    "constructor_prior_avg_finish": df["positionOrder"].mean(),
    "constructor_prior_podium_rate": df["podium"].mean(),
}
df = df.fillna(history_fill_values)

feature_cols = [
    "grid",
    "started_from_pit_or_back",
    "year",
    "round",
    "driver_starts_before",
    "constructor_starts_before",
    "driver_prior_avg_finish",
    "driver_prior_podium_rate",
    "constructor_prior_avg_finish",
    "constructor_prior_podium_rate",
]

model_df = df[feature_cols + ["podium", "driverRef", "constructorRef", "name"]].dropna().copy()
model_df.head()

## 3. Create a train/test split

I use a time-based split rather than a random split. The model trains on older seasons and is evaluated on newer seasons, which is closer to using historical performance to predict future races.

In [0]:
split_year = 2018

train_df = model_df[model_df["year"] < split_year].copy()
test_df = model_df[model_df["year"] >= split_year].copy()

X_train = train_df[feature_cols]
y_train = train_df["podium"]
X_test = test_df[feature_cols]
y_test = test_df["podium"]

print(f"Training rows: {len(train_df):,}")
print(f"Testing rows: {len(test_df):,}")
print(f"Train podium rate: {y_train.mean():.3f}")
print(f"Test podium rate: {y_test.mean():.3f}")

## 4. Set up MLflow

The experiment path uses the current Databricks user when possible. If that lookup is not available, it falls back to a shared workspace-style path.

In [0]:
try:
    current_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
    experiment_path = f"/Users/{current_user}/f1_podium_gradient_boosting"
except Exception:
    experiment_path = "/Shared/f1_podium_gradient_boosting"

mlflow.set_experiment(experiment_path)
print(experiment_path)

## 5. Run 10 tracked experiments

Each run logs the hyperparameters, the fitted model, several classification metrics, and three artifacts: a confusion matrix plot, an ROC curve plot, and a CSV of feature importances.

In [0]:
param_grid = [
    {"n_estimators": 50, "learning_rate": 0.03, "max_depth": 2, "subsample": 1.0, "min_samples_leaf": 1},
    {"n_estimators": 75, "learning_rate": 0.03, "max_depth": 2, "subsample": 0.8, "min_samples_leaf": 2},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2, "subsample": 1.0, "min_samples_leaf": 2},
    {"n_estimators": 125, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8, "min_samples_leaf": 3},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3, "subsample": 1.0, "min_samples_leaf": 3},
    {"n_estimators": 175, "learning_rate": 0.08, "max_depth": 2, "subsample": 0.8, "min_samples_leaf": 4},
    {"n_estimators": 200, "learning_rate": 0.08, "max_depth": 3, "subsample": 1.0, "min_samples_leaf": 4},
    {"n_estimators": 225, "learning_rate": 0.10, "max_depth": 2, "subsample": 0.8, "min_samples_leaf": 5},
    {"n_estimators": 250, "learning_rate": 0.10, "max_depth": 3, "subsample": 1.0, "min_samples_leaf": 5},
    {"n_estimators": 300, "learning_rate": 0.12, "max_depth": 2, "subsample": 0.8, "min_samples_leaf": 6},
]

run_summaries = []

In [0]:
for run_number, params in enumerate(param_grid, start=1):
    run_name = f"gb_podium_run_{run_number:02d}"

    with mlflow.start_run(run_name=run_name) as run:
        model = GradientBoostingClassifier(random_state=42, **params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "average_precision": average_precision_score(y_test, y_prob),
        }

        mlflow.log_param("model_type", "GradientBoostingClassifier")
        mlflow.log_param("target", "podium_finish")
        mlflow.log_param("split_year", split_year)
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        signature = infer_signature(X_train, model.predict_proba(X_train)[:, 1])
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)

        with tempfile.TemporaryDirectory() as tmpdir:
            cm_path = os.path.join(tmpdir, f"confusion_matrix_run_{run_number:02d}.png")
            fig, ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=False)
            ax.set_title(f"Podium Confusion Matrix: Run {run_number:02d}")
            fig.tight_layout()
            fig.savefig(cm_path, dpi=150)
            plt.close(fig)
            mlflow.log_artifact(cm_path, artifact_path="plots")

            roc_path = os.path.join(tmpdir, f"roc_curve_run_{run_number:02d}.png")
            fig, ax = plt.subplots(figsize=(5, 4))
            RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
            ax.set_title(f"ROC Curve: Run {run_number:02d}")
            fig.tight_layout()
            fig.savefig(roc_path, dpi=150)
            plt.close(fig)
            mlflow.log_artifact(roc_path, artifact_path="plots")

            feature_importance_path = os.path.join(tmpdir, f"feature_importance_run_{run_number:02d}.csv")
            feature_importance = (
                pd.DataFrame({"feature": feature_cols, "importance": model.feature_importances_})
                .sort_values("importance", ascending=False)
            )
            feature_importance.to_csv(feature_importance_path, index=False)
            mlflow.log_artifact(feature_importance_path, artifact_path="tables")

            report_path = os.path.join(tmpdir, f"classification_report_run_{run_number:02d}.txt")
            with open(report_path, "w") as f:
                f.write(classification_report(y_test, y_pred, zero_division=0))
            mlflow.log_artifact(report_path, artifact_path="tables")

        run_summary = {"run_name": run_name, "run_id": run.info.run_id, **params, **metrics}
        run_summaries.append(run_summary)

summary_df = pd.DataFrame(run_summaries).sort_values("f1", ascending=False)
summary_df

## 6. Select the best model run

For this assignment, I choose the best run primarily by F1 score. Podium finishes are much less common than non-podium finishes, so F1 is more informative than accuracy alone because it balances precision and recall.

In [0]:
best_run = summary_df.iloc[0]

print("Best run:", best_run["run_name"])
print("Run ID:", best_run["run_id"])
print(f"F1: {best_run['f1']:.3f}")
print(f"Precision: {best_run['precision']:.3f}")
print(f"Recall: {best_run['recall']:.3f}")
print(f"ROC AUC: {best_run['roc_auc']:.3f}")
print(f"Average precision: {best_run['average_precision']:.3f}")

best_run.to_frame("value")

## 7. Best run explanation

The next cell creates a written explanation using the metrics from the best tracked run. This way the explanation reflects the actual Databricks results after the experiment finishes.

In [0]:
best_model_explanation = f"""
My best model run was {best_run['run_name']}, a GradientBoostingClassifier trained to predict whether a driver would finish on the podium. I selected this run because it had the highest F1 score among the 10 tracked experiments. I used F1 as the main selection metric because podium finishes are the minority class, so accuracy by itself can overstate model quality.

The best run achieved an F1 score of {best_run['f1']:.3f}, precision of {best_run['precision']:.3f}, recall of {best_run['recall']:.3f}, ROC AUC of {best_run['roc_auc']:.3f}, and average precision of {best_run['average_precision']:.3f}. This combination suggests that the model produced the strongest overall balance between identifying actual podium finishers and avoiding too many false podium predictions.
"""

print(best_model_explanation)

## 8. Screenshot checklist for GitHub Classroom

After the experiment runs, take and upload screenshots of:

- the MLflow experiment homepage showing the 10 runs
- one detailed run page for the best run
- the logged model, metrics, parameters, and artifacts visible on that run page